### Kafka Topic Creation

This section creates a Kafka topic that will be used to stream the Spotify dataset
into the clustering pipeline.  

The topic is created using Kafka’s AdminClient API.
We configure it with **one partition**, which is sufficient for our **sequential
streaming experiment**, and a replication factor of **1** since the project runs
locally on a single Kafka broker.

The code also checks whether the topic already exists to avoid runtime errors
when the notebook is executed multiple times.

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic

def create_kafka_topic(topic_name):
    admin_client = AdminClient({"bootstrap.servers": "localhost:9092"})

    # Configure a simple, single-partition topic for local sequential experiments
    new_topic = NewTopic(topic_name, num_partitions=1, replication_factor=1)

    fs = admin_client.create_topics([new_topic])

    for topic, f in fs.items():
        try:
            # Block until topic creation completes (or fails)
            f.result()
            print(f" Topic '{topic}' created successfully and is ready!")
        except Exception as e:
            # Make the notebook idempotent by ignoring the "already exists" error
            if "TOPIC_ALREADY_EXISTS" in str(e):
                print(f" Topic '{topic}' already exists and is ready.")
            else:
                print(f" Failed to create topic '{topic}': {e}")

# Create the topic once at notebook startup
create_kafka_topic("final_runn")

 Topic 'final_runn' created successfully and is ready!


### Environment Setup and Imports

This section imports all required libraries for the streaming pipeline.

The notebook uses:
- **Kafka (confluent_kafka)** to consume streaming messages
- **Spark** for data processing and distributed centroid updates
- **NumPy / Pandas** for numerical work and final result summarization
- **uuid** to generate a unique Kafka consumer group ID

Using a unique consumer group prevents Kafka from waiting for old
inactive consumers and ensures the streaming pipeline starts immediately.

In [ ]:
"""### Environment Setup and Imports

This section imports all required libraries for the streaming pipeline.

The notebook uses:
- **Kafka (confluent_kafka)** to consume streaming messages
- **Spark** for data processing and distributed centroid updates
- **NumPy / Pandas** for numerical work and final result summarization
- **uuid** to generate a unique Kafka consumer group ID

Using a unique consumer group prevents Kafka from waiting for old
inactive consumers and ensures the streaming pipeline starts immediately.
"""

import json
import math
import time
import uuid
import numpy as np
import pandas as pd
import IPython.display as display
from confluent_kafka import Consumer, KafkaError
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StructType, StructField, DoubleType

# Note: scipy is not required for K-Means (unlike GMM),
# so we keep this import block clean and lightweight.

### Spark Session Initialization and Warm Start

This section initializes the **Apache Spark session** used to process the streaming
data received from Kafka and performs a **warm start** so that the first real
windows do not pay the full JVM and optimizer cost.

The code:
- creates the Spark session and sets the log level to **WARN** (to reduce log noise)
- runs a tiny Spark SQL job (`range(1, 100).count()`) to warm up basic execution
- builds a small dummy DataFrame and assembles it into a `features` vector
- fits a small K-Means model on this dummy data to load **MLlib K-Means and linear
  algebra libraries** into memory

After this warm start, the cluster is ready for true benchmarking and streaming
clustering without heavy one‑time initialization overhead.

In [ ]:
spark = SparkSession.builder.appName("KafkaConsumer-IncrementalKMeans-NoConnector").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

# Run a tiny SQL job to trigger query planning and JVM initialization
spark.range(1, 100).count()

# Create a minimal 1D dataset to warm up the ML pipeline
dummy_df = spark.createDataFrame([(1.0,), (2.0,), (10.0,), (11.0,)], ["val"])
assembler = VectorAssembler(inputCols=["val"], outputCol="features")
dummy_data = assembler.transform(dummy_df).select("features")

# Fit a very small K-Means once so that MLlib loads its native code and classes
KMeans(k=2, maxIter=2, seed=42, featuresCol="features").fit(dummy_data)
print("Warmup complete! The cluster is ready for true benchmarking.")

26/03/23 15:55:34 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
Spark version: 3.3.0


Warmup complete! The cluster is ready for true benchmarking.


### Kafka Consumer Configuration

This section initializes the Kafka consumer that will read streaming messages
from the Spotify topic.

A **unique consumer group ID** is generated using `uuid` to ensure that each
execution of the notebook starts a fresh consumer and avoids conflicts with
previous sessions.

The configuration also specifies `auto.offset.reset="earliest"` so that the
consumer can read all available messages if no previous offset exists.

In [ ]:

TOPIC_NAME = "final_runn"

# Configure a fresh consumer group for each run so offsets do not leak between experiments
conf = {
    "bootstrap.servers": "localhost:9092",
    "group.id": f"spotify-kmeans-group-{uuid.uuid4()}",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": True
}

consumer = Consumer(conf)
print(f"Kafka consumer created. Topic: {TOPIC_NAME}")

Kafka consumer created. Topic: final_runn


### Streaming and Clustering Configuration

This section defines the main parameters used in the streaming clustering pipeline.

The **window size** determines how often incoming Kafka data is processed in
micro-batches.
The **number of clusters (K_FIXED)** specifies how many groups the incremental
K-Means model will maintain over time.
A **random seed** is fixed to keep the initialization of the model reproducible.

In addition, several variables are initialized to maintain the **state of the
model across streaming windows**, including the centroids, cluster counts, and
the global feature order.

Note that in the current version, clustering quality is evaluated only once at
the end of each experiment using a **global silhouette score**, so the
`silhouette_history` list is kept for possible future extensions but is not
updated per window.

In [ ]:
# Length of each time-based window (in seconds) before processing the buffer
WINDOW_SECONDS = 2
# Fixed number of clusters used by the incremental K-Means model
K_FIXED = 2
# Seed used wherever randomness is involved (K-Means initialization)
SEED = 42

# Global model state that persists across windows
centroids = None          # Current cluster centers (updated incrementally)
cluster_counts = None     # How many points contributed to each centroid so far
global_feature_cols = []  # Canonical feature order shared by all windows
silhouette_history = []   # Reserved for optional per-window metrics (not used now)

### Feature Alignment and Data Preparation

Before applying the clustering algorithm, the data coming from each streaming
window must be converted into a consistent numerical format.

Since the structure of the data may slightly change between windows, the code
ensures that every batch has the **same feature layout**. Missing columns are
automatically created and filled with zeros so that the feature vectors remain
compatible with the existing clustering model, and all feature columns are cast
to `double`.

This alignment is done **directly in Spark**, keeping the data distributed
across workers until the Map–Reduce centroid update runs.

Additionally, if new features appear in later streaming windows, the centroid
vectors are automatically extended with zeros so that the model can continue
operating without losing alignment between the data and the cluster centroids.

In [ ]:
def _align_features_distributed(df, feature_cols):
    """
    Ensure that the window DataFrame has:
    - all expected feature columns
    - correct numeric type (double)
    - consistent column order.
    """
    existing = set(df.columns)
    for c in feature_cols:
        if c not in existing:
            # Add missing feature columns as zeros so vectors keep the same dimension
            df = df.withColumn(c, F.lit(0.0))
        # Force the column to double to avoid type mismatch in numerical code
        df = df.withColumn(c, F.col(c).cast("double"))
    # Fill any remaining nulls and enforce a stable column ordering
    return df.na.fill(0.0).select(*feature_cols)


def _extend_centroids_if_new_features(new_feature_cols):
    """
    Extend existing centroids when new feature columns appear in later windows.

    We keep the old dimensions untouched and pad each centroid with zeros
    for the new dimensions so that the model can continue to run without
    reinitializing from scratch.
    """
    global centroids, global_feature_cols

    if centroids is None:
        # Nothing to extend if the model is not initialized yet
        return

    old_cols = global_feature_cols
    old_set = set(old_cols)

    # Detect feature names that were not present in previous windows
    extra = [c for c in new_feature_cols if c not in old_set]
    if not extra:
        return

    # Horizontally stack zero columns for each new feature dimension
    pad = np.zeros((centroids.shape[0], len(extra)), dtype=float)
    centroids = np.hstack([centroids, pad])

    # Remember the updated full list of features
    global_feature_cols = old_cols + extra

### Silhouette Score Approximation for Final Evaluation

This function computes a **fast approximation of the silhouette score** based on
cluster centroids. The silhouette score measures how well each data point fits
within its assigned cluster compared to other clusters.

For each observation \(x\):

- \(a(x)\): distance between the point and its assigned centroid  
- \(b(x)\): distance to the nearest other centroid  
- \(s(x)\): silhouette score defined as \( (b - a) / \max(a, b) \)

In this notebook the function is used **once at the end of each experiment** on
the full dataset to produce a **global silhouette score**. This provides an
overall indicator of clustering quality without computing all pairwise
distances between points.

In [ ]:
def centroid_silhouette(X, centroids):
    """
    Approximate the global silhouette score using only distances to centroids.

    This avoids the O(n^2) cost of comparing every pair of samples and instead
    uses:
      - distance to the assigned centroid (a)
      - distance to the closest *other* centroid (b).
    """
    # Squared distances from each sample to each centroid: shape (N, K)
    d2 = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)

    # Index of the nearest centroid for each sample
    assign = d2.argmin(axis=1)

    # Distance to the assigned centroid a(x)
    a = np.sqrt(d2[np.arange(X.shape[0]), assign])

    # Distance to the closest *other* centroid b(x)
    d2_masked = d2.copy()
    d2_masked[np.arange(X.shape[0]), assign] = np.inf
    b = np.sqrt(d2_masked.min(axis=1))

    # Stabilize denominator to avoid division by zero for perfectly centered points
    denom = np.maximum(a, b)
    denom = np.where(denom == 0, 1e-12, denom)

    # Silhouette for each sample and mean over all samples
    s = (b - a) / denom
    return float(s.mean())

### Initial Centroid Computation using Spark K-Means

Before the incremental clustering process begins, the algorithm requires an initial
set of centroids. These centroids are computed using the **first streaming window**
and a standard **Spark ML K-Means** model.

All feature columns are first converted to numeric values and missing values are
replaced with zeros to ensure the dataset is suitable for clustering. The features
are then assembled into a single vector using Spark’s `VectorAssembler`, which is
required by the Spark ML library.

After fitting the K-Means model, the function extracts:
- the **initial cluster centroids**
- the **number of points assigned to each cluster**

These values are used to initialize the **incremental, distributed K-Means model**
that will be updated as new streaming windows arrive.

In [ ]:
def initialize_centroids_with_kmeans(df, k):
    """
    Initialize centroids using Spark KMeans on the first streaming window.

    We let Spark's implementation run K-Means++ on the first batch to get
    a good starting point, then switch to our own incremental updates.
    """
    feature_cols = df.columns
    if not feature_cols:
        raise ValueError("No features found in the first window.")

    wide = df
    # Convert every feature to double so it can be used by Spark ML
    for c in feature_cols:
        wide = wide.withColumn(c, col(c).cast("double"))
    # Replace missing values with zeros so vectors are dense and numeric
    wide = wide.na.fill(0.0)

    # Create a single feature vector column required by Spark ML
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    data = assembler.transform(wide).select("features")

    # Train initial K-Means model using Spark's optimized implementation
    model = KMeans(k=k, seed=SEED, featuresCol="features", predictionCol="prediction").fit(data)
    centers = np.array(model.clusterCenters(), dtype=float)

    # Count how many points Spark assigned to each cluster in the first window
    preds = model.transform(data).select("prediction")
    counts_rows = preds.groupBy("prediction").count().collect()
    counts = np.zeros((k,), dtype=float)
    for r in counts_rows:
        counts[int(r["prediction"])] = float(r["count"])

    return feature_cols, centers, counts

### Incremental Centroid Update (Distributed Map–Reduce)

After the initial centroids are computed, the clustering model is updated
incrementally for every new streaming window using a **true distributed
Map–Reduce approach**.

For each batch of data:

- Spark workers receive the current centroids via broadcast.
- Each partition (MAP phase) assigns its local observations to the nearest
  centroid and accumulates local **sums and counts** per cluster.
- The driver (REDUCE phase) aggregates these partial sums and counts using
  `treeAggregate`.
- The centroids are then updated on the driver using a **weighted running
  average**:

$$
\mu_{new} = \frac{n_{old}\mu_{old} + \sum_{window}}{n_{old} + n_{window}}
$$

where $n_{old}$ is the number of points previously assigned to the cluster and
$n_{window}$ is the number of new points assigned in the current window.

This design allows the clustering model to **adapt continuously to streaming
data** while keeping the heavy assignment and aggregation work distributed.

In [ ]:
def update_centroids_distributed(df_aligned, feature_cols):
    """
    Perform a distributed Map–Reduce update of K-Means centroids.

    MAP phase  : each partition assigns its rows to the nearest centroid
                 and accumulates local sums and counts.
    REDUCE phase: local results are aggregated on the driver to update
                  the global centroids with a running average.
    """
    global centroids, cluster_counts
    K, D = centroids.shape

    # Share the current centroids with all executors
    bc_centroids = spark.sparkContext.broadcast(centroids)

    def process_partition(iterator):
        """
        MAP: For rows in this partition, compute local cluster counts and sums.
        """

        local_counts = np.zeros(K, dtype=float)
        local_sums = np.zeros((K, D), dtype=float)

        for row in iterator:
            # Build a dense feature vector from the row
            x = np.array([row[c] for c in feature_cols], dtype=float)
            cents = bc_centroids.value
            # Squared distance to each centroid
            d2 = ((x - cents) ** 2).sum(axis=1)
            closest_k = int(d2.argmin())

            local_counts[closest_k] += 1.0
            local_sums[closest_k] += x

        yield (local_counts, local_sums)

    def reduce_partitions(a, b):
        """
        REDUCE: Merge two (counts, sums) tuples element-wise.
        """
        return (a[0] + b[0], a[1] + b[1])

    # Aggregate partial results from all partitions using a tree reduction
    summary = df_aligned.rdd.mapPartitions(process_partition).treeAggregate(
        (np.zeros(K, dtype=float), np.zeros((K, D), dtype=float)),
        reduce_partitions,
        reduce_partitions
    )

    window_counts, sum_x = summary

    # Update each centroid with a weighted average of previous and new data
    for k in range(K):
        if window_counts[k] > 0:
            total = cluster_counts[k] + window_counts[k]
            centroids[k] = ((cluster_counts[k] * centroids[k]) + sum_x[k]) / total
            cluster_counts[k] = total

    return window_counts

### Streaming Window Processing

This function manages the processing of each incoming streaming window.

For every window, the buffered Kafka records are converted into a Spark DataFrame
with an explicit numeric schema. If the model has not yet been initialized, the
first window is used to compute the initial centroids with Spark K-Means. For all
subsequent windows, the function:

- extends the centroid vectors if new feature columns appear
- aligns the incoming data with the global feature structure in a distributed way
- performs a **distributed Map–Reduce centroid update** using
  `update_centroids_distributed`
- logs the per-window and cumulative cluster counts for inspection

The design focuses on scalable, distributed updates rather than per-window
silhouette computation; clustering quality is assessed globally at the end of
each experiment.

In [ ]:
def process_window(records, window_id):
    """
    Convert the buffered Kafka records into a Spark DataFrame and either:
    - initialize the model on the first window, or
    - perform an incremental centroid update on subsequent windows.
    """
    global centroids, cluster_counts, global_feature_cols, silhouette_history

    if not records:
        # Nothing to process in this window
        return

    # Use the first record as a template to define a numeric schema
    fields = [StructField(c, DoubleType(), True) for c in records[0].keys()]
    schema = StructType(fields)
    df = spark.createDataFrame(records, schema=schema)

    new_cols = df.columns

    if centroids is None:
        # First window: bootstrap the incremental model using Spark ML K-Means++
        global_feature_cols, centroids, cluster_counts = initialize_centroids_with_kmeans(df, K_FIXED)
        print(f"[window {window_id}] Initialized K-Means | K={K_FIXED} | features={len(global_feature_cols)}")
        return

    # For later windows, adapt the model/features and apply the streaming update
    _extend_centroids_if_new_features(new_cols)
    df_aligned = _align_features_distributed(df, global_feature_cols)

    window_counts = update_centroids_distributed(df_aligned, global_feature_cols)

    # Keep the last window counts for potential debugging or analysis
    global last_window_counts
    last_window_counts = window_counts.copy()

    print(
        f"[window {window_id}] Updated K-Means | "
        f"window cluster counts={window_counts.astype(int).tolist()} | "
        f"total counts={cluster_counts.astype(int).tolist()}"
    )

### Kafka Streaming Consumption and Window Management

This function controls the main **Kafka streaming loop**.  
It continuously consumes messages from the specified Kafka topic and groups them
into time-based processing windows.

Each window collects records in a buffer and triggers processing when either:
- the predefined **window duration** is reached, or
- the buffer reaches a predefined **fast-forward size threshold**.

Once a window is ready, the buffered records are sent to the `process_window`
function, which performs the distributed centroid update.

The function also implements two additional mechanisms:

- **Benchmark tracking**: measures the total time required to process a fixed
  number of streaming windows.
- **Idle timeout protection**: terminates the loop early if the stream stops
  sending messages for a specified period.

This logic ensures that the streaming pipeline remains responsive and that
performance metrics can be collected during execution.

In [ ]:
def consume_loop_windowed(consumer, topic, target_windows=20):
    """
    Main streaming loop:
    - pulls JSON messages from Kafka,
    - groups them into time-based windows,
    - calls process_window for each completed window,
    - measures total runtime for a fixed number of windows.
    """
    consumer.subscribe([topic])
    print(f"Listening on topic: {topic} | Benchmark target: {target_windows} windows")

    records_buffer = []
    window_id = 0

    window_start = None          # timestamp when current window started
    first_message_time = None    # timestamp of first ever message (for total time)
    last_message_time = None     # timestamp of last message (for idle timeout)

    FAST_FORWARD_SIZE = 10000    # safety valve: process early if buffer grows too large
    IDLE_TIMEOUT = 10.0          # stop if no messages arrive for this many seconds

    try:
        while True:
            now = time.time()

            # Check if current window should be closed based on time or size
            if window_start is not None:
                time_is_up = (now - window_start) >= WINDOW_SECONDS
                buffer_is_full = len(records_buffer) >= FAST_FORWARD_SIZE

                if time_is_up or buffer_is_full:
                    if records_buffer:
                        window_id += 1
                        if buffer_is_full and not time_is_up:
                            print(f"[Speed Trigger] Fast-forwarding window {window_id} ({len(records_buffer)} rows)")

                        process_window(records_buffer, window_id)
                        records_buffer = []

                        # End the benchmark after the requested number of windows
                        if window_id == target_windows:
                            total_time = time.time() - first_message_time
                            print(f"BENCHMARK COMPLETE")
                            print(f"Target: {target_windows} windows")
                            print(f"Total Time: {total_time:.2f} seconds")
                            print(f"Average Time per Window: {total_time / target_windows:.4f} seconds")
                            return total_time

                    # Start measuring the next window
                    window_start = time.time()

            # If no messages arrive for too long, stop the experiment early
            if last_message_time is not None and (now - last_message_time) > IDLE_TIMEOUT:
                print(f" Stream went quiet for {IDLE_TIMEOUT} seconds. Ending early at window {window_id}.")
                total_time = (time.time() - first_message_time) - IDLE_TIMEOUT
                print(f" BENCHMARK COMPLETE (EARLY EXIT) ")
                print(f"Target: {target_windows} windows | Completed: {window_id} windows")
                print(f"Total Time: {total_time:.2f} seconds")
                return total_time

            # Poll Kafka for a new message (non-blocking with small timeout)
            msg = consumer.poll(timeout=0.1)

            if msg is None:
                continue

            last_message_time = time.time()

            if msg.error():
                # Ignore "end of partition" and print any real errors
                if msg.error().code() == KafkaError._PARTITION_EOF:
                    continue
                print("Kafka error:", msg.error())
                continue

            # On the very first message, start the benchmark timer and first window
            if window_start is None:
                window_start = time.time()
                first_message_time = window_start
                last_message_time = window_start
                print("First message received! Starting the benchmark timer...")

            # Decode JSON payload and append it to the current buffer
            try:
                payload = msg.value().decode("utf-8")
                obj = json.loads(payload)
            except Exception as e:
                print("Bad JSON message, skipping. Error:", e)
                continue

            if isinstance(obj, dict):
                records_buffer.append(obj)
            elif isinstance(obj, list):
                # Allow producers to send batches (lists of dicts) in one Kafka message
                records_buffer.extend([x for x in obj if isinstance(x, dict)])
            else:
                # Ignore any unexpected message shape
                continue

    except KeyboardInterrupt:
        print("\nStopping (KeyboardInterrupt).")
        # If we had started timing, return the elapsed time
        if first_message_time:
             return time.time() - first_message_time
        return 0

    finally:
        # If there is a partial window left, process it once before closing
        if records_buffer and window_id < target_windows:
            window_id += 1
            process_window(records_buffer, window_id)
        print("Finished consuming current dataset.")

### Model State Reset

This utility function resets the global state of the clustering model before
starting a new streaming experiment.

All variables that store the model state (centroids, cluster counts, feature
structure, and evaluation history) are cleared. This ensures that each dataset
run starts from a **clean initialization**, preventing interference from
previous experiments.

In [ ]:
def reset_model_state():
    """
    Reset all global variables that store the clustering model state.

    This is called before each experiment so runs do not interfere with
    each other (no centroids or counts are carried over).
    """
    global centroids, cluster_counts, global_feature_cols
    global silhouette_history, last_window_counts

    centroids = None
    cluster_counts = None
    global_feature_cols = []
    silhouette_history = []
    last_window_counts = None
    print(" Model state has been completely reset.")

### Global Silhouette Evaluation on the Full Dataset

After the streaming experiment is completed, this function evaluates the final
clustering quality using the **entire dataset**.

The dataset is loaded directly from the CSV file that was used for streaming.
All features are aligned with the global feature structure using the same
distributed alignment function as during streaming. The data is then collected
into NumPy **only once** for this final evaluation, and the final centroids
obtained from the incremental clustering process are used to compute a
**global silhouette score**.

This metric provides a final estimate of clustering quality across the whole
dataset and complements the per-window logging of cluster counts.

In [ ]:
def evaluate_full_dataset_silhouette(spark, final_centroids, feature_cols, csv_path):
    """
    Load the full dataset from CSV, align it to the model features,
    and compute a single global silhouette score with the final centroids.
    """

    if final_centroids is None:
        # No model was trained, nothing to evaluate
        return None

    # Load the offline CSV that corresponds to this experiment
    df_full = spark.read.csv(csv_path, header=True, inferSchema=True)

    # Apply the same feature alignment logic used during streaming
    df_aligned = _align_features_distributed(df_full, feature_cols)

    # Collect to the driver only once per experiment for metric calculation
    rows = df_aligned.collect()
    if not rows:
        return None

    X_full = np.array([[r[c] for c in feature_cols] for r in rows], dtype=float)

    global_score = centroid_silhouette(X_full, final_centroids)
    return global_score

### Streaming Experiments Execution and Benchmarking

This section runs the complete streaming clustering pipeline on multiple datasets
in order to evaluate the performance and scalability of the system.

A list of datasets is defined, each specifying the file path, the number of rows,
and the batch size used during streaming. For each experiment, the pipeline
performs the following steps:

1. **Reset the model state** to ensure that each experiment starts from a clean configuration.
2. **Consume and process streaming windows** using the Kafka consumer and the
   windowed streaming loop with distributed centroid updates.
3. **Evaluate the final clustering quality** by computing a single global silhouette score
   on the entire dataset.
4. **Store performance metrics**, including total processing time and global clustering quality.

The results from all experiments are aggregated into a summary table, allowing
easy comparison between datasets of different sizes.

In [ ]:
# Define the datasets used for the experiments (each row count determines windows)
experiments = [
    {"file": "exported_dataframes/df_30k_part_1.csv", "rows": 30000, "batch_size": 10000},
    {"file": "exported_dataframes/df_30k_part_2.csv", "rows": 30000, "batch_size": 10000},
    {"file": "exported_dataframes/df_30k_part_3.csv", "rows": 30000, "batch_size": 10000},
    {"file": "exported_dataframes/df_60k_part_1.csv", "rows": 60000, "batch_size": 10000},
    {"file": "exported_dataframes/df_60k_part_2.csv", "rows": 60000, "batch_size": 10000},
    {"file": "exported_dataframes/df_60k_part_3.csv", "rows": 60000, "batch_size": 10000},
    {"file": "exported_dataframes/df_90k_part_1.csv", "rows": 90000, "batch_size": 10000},
    {"file": "exported_dataframes/df_90k_part_2.csv", "rows": 90000, "batch_size": 10000},
    {"file": "exported_dataframes/df_90k_part_3.csv", "rows": 90000, "batch_size": 10000},
    {"file": "exported_dataframes/spotify_data_prepped.csv", "rows": 113999, "batch_size": 10000}
]

TOPIC_NAME = "final_runn"
experiment_results = []

for exp in experiments:
    file_path = exp["file"]

    # Compute how many windows we expect, given rows and batch size
    target_windows = math.ceil(exp["rows"] / exp["batch_size"])

    print("\n" + "=" * 70)
    print(f"STARTING EXPERIMENT: {file_path} ({exp['rows']} rows)")
    print("=" * 70)

    # Start from a clean model so results for each dataset are independent
    reset_model_state()

    # Run the streaming benchmark for this dataset (data is already in Kafka)
    total_time = consume_loop_windowed(consumer, TOPIC_NAME, target_windows)

    # Compute final clustering quality on the full offline CSV
    global_score = evaluate_full_dataset_silhouette(spark, centroids, global_feature_cols, file_path)
    print(f" Global Silhouette Score for {file_path}: {global_score:.6f}")

    # Record performance and quality metrics for later comparison
    experiment_results.append({
        "Dataset": file_path,
        "Rows": exp["rows"],
        "Target Windows": target_windows,
        "Total Time (sec)": round(total_time, 2),
        "Global Silhouette": round(global_score, 6)
    })

# Display a summary DataFrame of all experiments
print("\n ALL EXPERIMENTS COMPLETE ")
results_df = pd.DataFrame(experiment_results)
display.display(results_df)

# Close the Kafka consumer once all experiments have finished
consumer.close()
print("Kafka consumer permanently closed.")



STARTING EXPERIMENT: exported_dataframes/df_30k_part_1.csv (30000 rows)
 Model state has been completely reset.
Listening on topic: final_runn | Benchmark target: 3 windows
First message received! Starting the benchmark timer...
26/03/23 15:55:52 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[window 1] Initialized K-Means | K=2 | features=15
[Speed Trigger] Fast-forwarding window 2 (10000 rows)
[window 2] Updated K-Means | window cluster counts=[6330, 3670] | total counts=[10804, 6196]
[window 3] Updated K-Means | window cluster counts=[5072, 2928] | total counts=[15876, 9124]
BENCHMARK COMPLETE
Target: 3 windows
Total Time: 8.62 seconds
Average Time per Window: 2.8745 seconds
Finished consuming current dataset.
 Global Silhouette Score for exported_dataframes/df_30k_part_1.csv: 0.366324

STARTING EXPERIMENT: exported_dataframes/df_30k_part_2.csv (30000 rows)
 Model stat

First message received! Starting the benchmark timer...
[Speed Trigger] Fast-forwarding window 1 (10000 rows)
[window 1] Initialized K-Means | K=2 | features=15
[Speed Trigger] Fast-forwarding window 2 (10000 rows)
[window 2] Updated K-Means | window cluster counts=[3580, 6420] | total counts=[7112, 12888]
[window 3] Updated K-Means | window cluster counts=[2576, 4424] | total counts=[9688, 17312]
[window 4] Updated K-Means | window cluster counts=[2841, 5159] | total counts=[12529, 22471]
[window 5] Updated K-Means | window cluster counts=[3010, 4990] | total counts=[15539, 27461]
[window 6] Updated K-Means | window cluster counts=[2494, 4506] | total counts=[18033, 31967]
[window 7] Updated K-Means | window cluster counts=[2888, 5112] | total counts=[20921, 37079]
[window 8] Updated K-Means | window cluster counts=[2939, 5061] | total counts=[23860, 42140]
[window 9] Updated K-Means | window cluster counts=[2491, 4509] | total counts=[26351, 46649]
BENCHMARK COMPLETE
Target: 9 window

,Dataset,Rows,Target Windows,Total Time (sec),Global Silhouette
0,exported_dataframes/df_30k_part_1.csv,30000,3,8.62,0.366324
1,exported_dataframes/df_30k_part_2.csv,30000,3,8.21,0.367731
2,exported_dataframes/df_30k_part_3.csv,30000,3,8.12,0.366696
3,exported_dataframes/df_60k_part_1.csv,60000,6,15.33,0.235600
4,exported_dataframes/df_60k_part_2.csv,60000,6,14.36,0.366783
5,exported_dataframes/df_60k_part_3.csv,60000,6,13.74,0.366798
6,exported_dataframes/df_90k_part_1.csv,90000,9,21.74,0.366704
7,exported_dataframes/df_90k_part_2.csv,90000,9,20.59,0.366353
8,exported_dataframes/df_90k_part_3.csv,90000,9,22.58,0.366561
9,exported_dataframes/spotify_data_prepped.csv,113999,12,28.03,0.234197


Kafka consumer permanently closed.
